In [ ]:
# LAEI 2019 modelling walkthrough

This notebook follows the implementation sequence in `Task1_Task2_Report`.

The paths assume that the notebook is saved in the project's `notebooks/` directory. The project data are loaded from `../data/processed/`, and the shared helper module is loaded from `../src/`.


## 1. Load the prepared data

The first cell checks that the project structure is correct. `sys.path.append` allows Python to find the project's local `laei.py` module. The CSV contains one row per road link, while the manifest records the agreed table definitions, feature counts, targets, and modelling protocol.

The expected link-table shape is `(79388, 56)`. This is a diagnostic only: it does not fit a model.


In [1]:
import json
import sys

import pandas as pd

# The notebook is expected to run from the project's `notebooks` directory.
# Adding `../src` makes the local project helper importable.
sys.path.append("../src")
import laei  # the project's shared helper module — short, worth reading

# Load the prepared road-link table. `low_memory=False` helps pandas infer
# mixed columns consistently rather than guessing in separate file chunks.
links = pd.read_csv(
    "../data/processed/table_A_links.csv",
    low_memory=False,
)

# The manifest is metadata, not a modelling table. It documents the
# intended feature sets, targets, and evaluation protocol.
with open("../data/processed/manifest.json", encoding="utf-8") as file:
    manifest = json.load(file)

print(links.shape)
print(manifest["tables"].keys()) # what has been prepared for modelling


(79388, 56)
dict_keys(['table_A_links.csv', 'table_B_grid_source_mix.csv', 'table_C_grid_nonleaky.csv', 'table_C_LEAKY_reference.csv'])


## 2. Fit a first realistic regression model

This deliberately small example confirms that the environment, paths, helper module, data, and scikit-learn pipeline all work together. It uses the realistic feature set, which excludes VKM because VKM leaks information about how the inventory target was calculated.

The pipeline fits preprocessing only on the training data. This is important: imputers and encoders must not learn information from the held-out test data.


In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# The helper returns two numeric feature lists: `full` (including VKM)
# and `realistic` (excluding VKM). The realistic set is the honest
# headline evaluation for a prediction task.
feature_sets = laei.link_feature_sets(links)
numeric_features = feature_sets["realistic"]
categorical_features = laei.LINK_CATEGORICAL

# X contains predictors; y is the NOx target in tonnes per year.
X = links[numeric_features + categorical_features]
y = links["nox"]

# Hold out 20% for an evaluation that is not used to fit the model.
# Fixing the seed makes the split reproducible.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

model = Pipeline(
    steps=[
        ("pre", laei.make_preprocessor(numeric_features, categorical_features)),
        ("model", RandomForestRegressor(
            n_estimators=200,
            n_jobs=-1,
            random_state=42,
        )),
    ]
).fit(X_train, y_train)

# Evaluate only on rows not used during fitting. R2, MAE, and RMSE
# describe complementary aspects of the skewed target.
print(laei.regression_metrics(y_test, model.predict(X_test)))


{'R2': 0.9871322627601798, 'MAE': 0.019051161001476895, 'RMSE': 0.12330745166595102}


## 3. Link-level regression

This section compares four regressors under two feature sets. The `full` set includes VKM and is a labelled leakage demonstration. The `realistic` set excludes VKM and is the appropriate basis for honest predictive performance.

The dummy baseline predicts the training median for every test row. It provides a reference against which learned models should be judged.


In [3]:
import os
import sys
import time

import numpy as np
import pandas as pd

# Limit joblib's CPU-count warning on Windows before parallel fitting starts.
os.environ["LOKY_MAX_CPU_COUNT"] = "8"

sys.path.append("../src")
import laei

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

links = pd.read_csv(
    "../data/processed/table_A_links.csv",
    low_memory=False,
)
feature_sets = laei.link_feature_sets(links)
CATEGORICAL = laei.LINK_CATEGORICAL
y = links["nox"]


def models():
    """Return fresh estimator objects for each modelling loop."""
    return {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(alpha=1.0, random_state=42),
        "RandomForest": RandomForestRegressor(
            n_estimators=200,
            n_jobs=-1,
            random_state=42,
        ),
        "HistGradientBoosting": HistGradientBoostingRegressor(
            max_iter=300,
            random_state=42,
        ),
    }


rows = []

# The loop deliberately runs both feature sets. Do not report the full-set
# score as honest performance: VKM is closely related to target construction.
for feature_set_name, numeric_features in feature_sets.items():
    X = links[numeric_features + CATEGORICAL]
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )

    # A median baseline ignores X and establishes how much a model improves
    # over a simple constant prediction.
    dummy = DummyRegressor(strategy="median").fit(X_train, y_train)
    print(
        feature_set_name,
        "baseline:",
        laei.regression_metrics(y_test, dummy.predict(X_test)),
    )

    for model_name, estimator in models().items():
        start = time.time()
        pipeline = Pipeline(
            steps=[
                ("pre", laei.make_preprocessor(numeric_features, CATEGORICAL)),
                ("model", estimator),
            ]
        ).fit(X_train, y_train)

        metrics = laei.regression_metrics(
            y_test,
            pipeline.predict(X_test),
        )
        rows.append({
            "feature_set": feature_set_name,
            "model": model_name,
            **metrics,
            "fit_s": round(time.time() - start, 1),
        })
        print(
            f"  {model_name:<21} "
            f"R2={metrics['R2']:.4f} "
            f"MAE={metrics['MAE']:.4f} "
            f"RMSE={metrics['RMSE']:.4f}"
        )

results = pd.DataFrame(rows)


full baseline: {'R2': -0.026007804028763015, 'MAE': 0.22146965053142095, 'RMSE': 1.1010665333460372}
  LinearRegression      R2=0.9907 MAE=0.0334 RMSE=0.1048
  Ridge                 R2=0.9907 MAE=0.0332 RMSE=0.1047
  RandomForest          R2=0.9888 MAE=0.0150 RMSE=0.1152
  HistGradientBoosting  R2=0.9502 MAE=0.0244 RMSE=0.2426
realistic baseline: {'R2': -0.026007804028763015, 'MAE': 0.22146965053142095, 'RMSE': 1.1010665333460372}
  LinearRegression      R2=0.7503 MAE=0.2060 RMSE=0.5432
  Ridge                 R2=0.7503 MAE=0.2060 RMSE=0.5432
  RandomForest          R2=0.9871 MAE=0.0191 RMSE=0.1233
  HistGradientBoosting  R2=0.9114 MAE=0.0445 RMSE=0.3236


## 4. Five-fold cross-validation

A single train/test split can be unusually favourable or unfavourable. Five-fold cross-validation repeats the train/test process five times, with each fold used once for validation.

`shuffle=True` is important because the rows are ordered approximately by geography. The reported mean and standard deviation describe both average performance and variation between folds.


In [4]:
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

for feature_set_name, numeric_features in feature_sets.items():
    for model_name, estimator in models().items():
        pipeline = Pipeline(
            steps=[
                ("pre", laei.make_preprocessor(numeric_features, CATEGORICAL)),
                ("model", estimator),
            ]
        )
        scores = cross_val_score(
            pipeline,
            links[numeric_features + CATEGORICAL],
            y,
            cv=cv,
            scoring="r2",
        )
        print(
            f"{feature_set_name:>10} {model_name:<21} "
            f"{scores.mean():.4f} ± {scores.std():.4f}"
        )


      full LinearRegression      0.9867 ± 0.0026
      full Ridge                 0.9867 ± 0.0026
      full RandomForest          0.9904 ± 0.0017
      full HistGradientBoosting  0.9626 ± 0.0113
 realistic LinearRegression      0.7561 ± 0.0124
 realistic Ridge                 0.7561 ± 0.0124
 realistic RandomForest          0.9870 ± 0.0013
 realistic HistGradientBoosting  0.9030 ± 0.0162


## 5. Borough-grouped cross-validation

Ordinary random folds can place links from the same borough in both training and validation data. `GroupKFold` prevents that by keeping every borough together.

This is a spatial-correlation sanity check: it asks how well the realistic random forest predicts links in boroughs that were absent from training.


In [5]:
from sklearn.model_selection import GroupKFold

numeric_features = feature_sets["realistic"]
pipeline = Pipeline(
    steps=[
        ("pre", laei.make_preprocessor(numeric_features, CATEGORICAL)),
        ("model", RandomForestRegressor(
            n_estimators=200,
            n_jobs=-1,
            random_state=42,
        )),
    ]
)

scores = cross_val_score(
    pipeline,
    links[numeric_features + CATEGORICAL],
    y,
    cv=GroupKFold(n_splits=5),
    groups=links["Borough"],
    scoring="r2",
)
print(f"by borough: {scores.mean():.4f}")


by borough: 0.9727


## 6. Feature importance

Impurity importance is extracted from the fitted forest, but it can favour continuous variables with many possible split points. Permutation importance provides a complementary test: it measures how much held-out R² falls when one feature is shuffled.

The permutation calculation is deliberately restricted to 6,000 test rows and repeated five times to reduce computation while retaining a stable ranking.


In [6]:
from sklearn.inspection import permutation_importance

X_train, X_test, y_train, y_test = train_test_split(
    links[numeric_features + CATEGORICAL],
    y,
    test_size=0.2,
    random_state=42,
)

forest = Pipeline(
    steps=[
        ("pre", laei.make_preprocessor(numeric_features, CATEGORICAL)),
        ("model", RandomForestRegressor(
            n_estimators=200,
            n_jobs=-1,
            random_state=42,
        )),
    ]
).fit(X_train, y_train)

# The fitted one-hot encoder supplies names for the transformed categories.
encoder = forest.named_steps["pre"].named_transformers_["cat"]
feature_names = numeric_features + list(
    encoder.get_feature_names_out(CATEGORICAL)
)
impurity = pd.Series(
    forest.named_steps["model"].feature_importances_,
    index=feature_names,
).sort_values(ascending=False)

sample = X_test.sample(6000, random_state=42)
permutation = permutation_importance(
    forest,
    sample,
    y_test.loc[sample.index],
    n_repeats=5,
    random_state=42,
    scoring="r2",
)
permutation_series = pd.Series(
    permutation.importances_mean,
    index=numeric_features + CATEGORICAL,
).sort_values(ascending=False)

display(impurity.head(10))
display(permutation_series.head(10))


Link Length (m)                                  0.686884
AADT 2019 - Total                                0.129500
AADT Diesel LGV                                  0.039415
AADT Petrol Car                                  0.033166
AADT Diesel Car                                  0.020921
AADT Diesel PHV                                  0.012562
AADT 2019 - HGVs - Articulated - 3 to 4 Axles    0.010470
AADT Petrol PHV                                  0.009643
AADT Electric Car                                0.008139
AADT Petrol LGV                                  0.006586
dtype: float64

Link Length (m)                                  1.015516
AADT 2019 - Total                                0.103179
AADT Petrol Car                                  0.013130
AADT Diesel LGV                                  0.011984
AADT Diesel Car                                  0.006403
Speed (km/hr) - Except Buses                     0.005983
AADT 2019 - HGVs - Articulated - 5 Axles         0.005885
AADT Petrol PHV                                  0.004283
AADT Diesel PHV                                  0.003732
AADT 2019 - HGVs - Articulated - 3 to 4 Axles    0.003259
dtype: float64

## 7. K-Means clustering

The matrix is already standardised and contains the 16 source-share features. Coordinates are retained in the source table for validation and plotting, but must not be used as clustering inputs.

The scan reports both inertia and silhouette. Inertia always decreases as k increases, so it cannot by itself identify the appropriate number of clusters. Low silhouette values indicate overlapping profiles; k=4 is retained for interpretability.


In [7]:
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

table_b = pd.read_csv(
    "../data/processed/table_B_grid_source_mix.csv"
)

X = np.load(
    "../data/processed/table_B_kmeans_matrix.npy"
)

for k in range(2, 11):
    kmeans = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=42,
    ).fit(X)
    print(
        f"k={k} inertia={kmeans.inertia_:>8.0f} "
        f"silhouette={silhouette_score(X, kmeans.labels_):.3f}"
    )

kmeans = KMeans(
    n_clusters=4,
    n_init=10,
    random_state=42,
).fit(X)
table_b["cluster"] = kmeans.labels_

share_columns = [
    column
    for column in table_b.columns
    if column.startswith("share_")
]
profile = table_b.groupby("cluster")[share_columns].mean() * 100
print(
    profile.round(1)
    .T
    .sort_values(0, ascending=False)
    .head(8)
)

# Coordinates are used only after fitting, to test whether cluster labels
# show spatial coherence without having used geography as a feature.
from sklearn.neighbors import NearestNeighbors

coordinates = table_b[["Easting", "Northing"]].to_numpy()
_, indices = NearestNeighbors(n_neighbors=9).fit(coordinates).kneighbors(
    coordinates
)
neighbours = indices[:, 1:]
labels = table_b["cluster"].to_numpy()
observed = (labels[neighbours] == labels[:, None]).mean()

rng = np.random.default_rng(42)
null = np.array([
    (lambda shuffled: (shuffled[neighbours] == shuffled[:, None]).mean())(
        rng.permutation(labels)
    )
    for _ in range(200)
])
print(
    f"neighbours agree {observed:.3f} vs {null.mean():.3f} shuffled, "
    f"z={(observed - null.mean()) / null.std():.1f}"
)

# Compare K-Means with a different clustering assumption. ARI = 1 means
# identical partitions; ARI around 0 indicates agreement no better than chance.
agglomerative = AgglomerativeClustering(n_clusters=4).fit(X)
print(
    "ARI vs KMeans:",
    round(adjusted_rand_score(kmeans.labels_, agglomerative.labels_), 3),
)
print(table_b["cluster"].value_counts().sort_index())


k=2 inertia=   33808 silhouette=0.250
k=3 inertia=   30482 silhouette=0.272
k=4 inertia=   27331 silhouette=0.272
k=5 inertia=   24545 silhouette=0.271
k=6 inertia=   21343 silhouette=0.284
k=7 inertia=   19395 silhouette=0.255
k=8 inertia=   16416 silhouette=0.278
k=9 inertia=   13940 silhouette=0.299
k=10 inertia=   11973 silhouette=0.321
cluster                             0     1     2     3
share_Aviation                   63.9   0.3   0.2   1.6
share_Road Transport             18.0  26.9  68.0   8.8
share_Heat and Power Generation  13.4  56.6  25.0  12.7
share_Waste                       1.4   0.1   0.2   0.4
share_Construction                1.1   1.8   3.0   0.9
share_Industrial Processes        1.0   3.6   1.1   3.6
share_River                       0.9   0.5   0.2  71.5
share_Agriculture                 0.2   6.9   0.9   0.5
neighbours agree 0.588 vs 0.508 shuffled, z=22.7
ARI vs KMeans: 0.385
cluster
0      69
1    1253
2    2071
3      67
Name: count, dtype: int64


## 8. Hotspot classification

The target is imbalanced: 346 of 3,460 cells are hotspots. A dummy classifier can therefore achieve high accuracy simply by predicting the majority class. Precision, recall, F1, and ROC-AUC are more informative.

The models use CO2-by-sector features and coordinates, not NOx-by-sector features, to avoid directly reconstructing the target.


In [8]:
import json

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

table_c = pd.read_csv(
    "../data/processed/table_C_grid_nonleaky.csv"
)
with open("../data/processed/manifest.json", encoding="utf-8") as file:
    manifest = json.load(file)
features = manifest["tables"]["table_C_grid_nonleaky.csv"]["features"]

X = table_c[features]
y = table_c["nox_hotspot"]

# Stratification preserves the approximately 10% hotspot rate in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=42,
)

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(
    "dummy:",
    laei.classification_metrics(y_test, dummy.predict(X_test)),
)

models = {
    "LogisticRegression": Pipeline(
        steps=[
            ("sc", StandardScaler()),
            ("m", LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=42,
            )),
        ]
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=400,
        n_jobs=-1,
        class_weight="balanced",
        random_state=42,
    ),
}

probabilities = {}
for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    probabilities[name] = estimator.predict_proba(X_test)[:, 1]
    print(
        name,
        laei.classification_metrics(
            y_test,
            estimator.predict(X_test),
            probabilities[name],
        ),
    )


dummy: {'precision': 0.0, 'recall': 0.0, 'F1': 0.0, 'accuracy': 0.900578034682081}
LogisticRegression {'precision': 0.5985401459854015, 'recall': 0.9534883720930233, 'F1': 0.7354260089686099, 'accuracy': 0.9317919075144508, 'ROC_AUC': 0.9815655133295519, 'avg_precision': 0.9169786176497642}
RandomForest {'precision': 0.9655172413793104, 'recall': 0.6511627906976745, 'F1': 0.7777777777777778, 'accuracy': 0.9630057803468208, 'ROC_AUC': 0.9888198943188943, 'avg_precision': 0.9160069897005777}


## 9. Cost-based threshold selection

A probability is not yet an inspection decision. The threshold converts a probability into a class label. Here, missing a genuine hotspot is assigned five times the cost of a false alarm.

The search tests thresholds from 0.05 to 0.95 in increments of 0.05 and selects the lowest total cost. The result is model-specific because probability scales differ between models.


In [9]:
COST_MISS, COST_FALSE_ALARM = 5.0, 1.0

for name, probabilities_for_model in probabilities.items():
    best = None
    for threshold in np.arange(0.05, 0.96, 0.05):
        true_negative, false_positive, false_negative, true_positive = (
            confusion_matrix(
                y_test,
                (probabilities_for_model >= threshold).astype(int),
                labels=[0, 1],
            ).ravel()
        )
        cost = (
            COST_MISS * false_negative
            + COST_FALSE_ALARM * false_positive
        )
        if best is None or cost < best[1]:
            best = (threshold, cost, false_negative, false_positive)
    print(
        f"{name}: best threshold {best[0]:.2f}, "
        f"cost {best[1]:.0f} "
        f"(missed {best[2]}, false alarms {best[3]})"
    )

best_thresholds = {
    "LogisticRegression": 0.65,
    "RandomForest": 0.25,
}
for name, probabilities_for_model in probabilities.items():
    threshold = best_thresholds[name]
    predictions = (probabilities_for_model >= threshold).astype(int)
    print(
        name,
        "at threshold",
        f"{threshold:.2f}:",
        laei.classification_metrics(
            y_test,
            predictions,
            probabilities_for_model,
        ),
    )


LogisticRegression: best threshold 0.65, cost 64 (missed 5, false alarms 39)
RandomForest: best threshold 0.25, cost 63 (missed 5, false alarms 38)
LogisticRegression at threshold 0.65: {'precision': 0.675, 'recall': 0.9418604651162791, 'F1': 0.7864077669902912, 'accuracy': 0.9491329479768786, 'ROC_AUC': 0.9815655133295519, 'avg_precision': 0.9169786176497642}
RandomForest at threshold 0.25: {'precision': 0.680672268907563, 'recall': 0.9418604651162791, 'F1': 0.7902439024390244, 'accuracy': 0.9502890173410404, 'ROC_AUC': 0.9888198943188943, 'avg_precision': 0.9160069897005777}


## 10. Intensity target

Total NOx measures quantity. `nox_per_m` measures intensity per metre and may be more relevant to roadside exposure. The following cell prepares the alternative target and then fits a model, producing a hold‑out evaluation for intensity.


In [22]:
mask = links["nox_per_m"].notna()
X_intensity = links.loc[mask, numeric_features + CATEGORICAL]
y_intensity = links.loc[mask, "nox_per_m"]

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import time
import pandas as pd

# Reuse the same model factory as in section 6.1
def models():
    from sklearn.linear_model import LinearRegression, Ridge
    from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

    return {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(alpha=1.0, random_state=42),
        "RandomForest": RandomForestRegressor(
            n_estimators=200,
            n_jobs=-1,
            random_state=42,
        ),
        "HistGradientBoosting": HistGradientBoostingRegressor(
            max_iter=300,
            random_state=42,
        ),
    }

# Train/test split for the intensity target
X_tr, X_te, y_tr, y_te = train_test_split(
    X_intensity,
    y_intensity,
    test_size=0.2,
    random_state=42,
)

rows_intensity = []

for name, estimator in models().items():
    t0 = time.time()
    pipe = Pipeline([
        ("pre", laei.make_preprocessor(numeric_features, CATEGORICAL)),
        ("model", estimator),
    ]).fit(X_tr, y_tr)

    met = laei.regression_metrics(y_te, pipe.predict(X_te))
    rows_intensity.append({
        "model": name,
        **met,
        "fit_s": round(time.time() - t0, 1),
    })
    print(
        f"  {name:<21} "
        f"R2={met['R2']:.4f} "
        f"MAE={met['MAE']:.4f} "
        f"RMSE={met['RMSE']:.4f}"
    )

results_intensity = pd.DataFrame(rows_intensity)

  LinearRegression      R2=0.9124 MAE=0.0003 RMSE=0.0006
  Ridge                 R2=0.9125 MAE=0.0003 RMSE=0.0006
  RandomForest          R2=0.9899 MAE=0.0001 RMSE=0.0002
  HistGradientBoosting  R2=0.9692 MAE=0.0002 RMSE=0.0004


## 11. Five-fold cross-validation for intensity

A single train/test split can be unusually favourable or unfavourable. Five-fold cross-validation repeats the train/test process five times, with each fold used once for validation.

`shuffle=True` is important because the rows are ordered approximately by geography. The reported mean and standard deviation describe both average performance and variation between folds.


In [29]:
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, estimator in models().items():
    pipe = Pipeline([
        ("pre", laei.make_preprocessor(numeric_features, CATEGORICAL)),
        ("model", estimator),
    ])
    scores = cross_val_score(
        pipe,
        X_intensity,
        y_intensity,
        cv=cv,
        scoring="r2",
    )
    print(
        f"{name:<21} {scores.mean():.4f} ± {scores.std():.4f}"
    )

LinearRegression      0.9096 ± 0.0023
Ridge                 0.9096 ± 0.0023
RandomForest          0.9894 ± 0.0009
HistGradientBoosting  0.9698 ± 0.0022
